# Turn-taking workshop: when should a voice assistant talk?

A voice assistant must decide, every moment, what to do with your speech:

| label | what you did | what the assistant should do |
|---|---|---|
| **respond** | you finished a sentence or question | answer now |
| **keep_going** | you paused in the middle of a sentence | wait, you are still talking |
| **stop** | you said "stop", "ruko", "thamba", "nillisu" | stop talking at once |

We will build this classifier in about an hour, on a normal laptop, with no GPU.

**The plan**
1. Listen to some example clips.
2. Turn each clip into numbers with a *frozen* speech model (whisper-tiny).
3. Train a simple classifier on those numbers (logistic regression).
4. Train a tiny neural network (MLP) and compare.
5. Test it live with your own voice.

Run each cell with **Shift + Enter**. Read the text before each cell.

In [ ]:
# Setup: find the project folder and load our helper functions.
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "common.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import common

print("Project folder:", ROOT)
print("Labels:", common.LABELS)

## Step 1. Listen to the data

The starter clips were made with text-to-speech in Kannada, Hindi, Marathi and English.
A **keep_going** clip is the same sentence as a **respond** clip, but cut in the middle.
So the model must learn *how* a sentence ends, not just *what* was said.

File names look like `kn_ttssapna_3.wav` = language `kn`, speaker `ttssapna`, clip number 3.

In [ ]:
clips = common.list_clips()
print("Total clips:", len(clips))

for label in common.LABELS:
    path, _, lang = next(c for c in clips if c[1] == label and c[2] == "hi")
    print(label, "-", path.name)
    display(Audio(str(path)))

Let us look at the waveforms. Notice where the speech ends and the silence begins.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for ax, label in zip(axes, common.LABELS):
    path = next(c[0] for c in clips if c[1] == label and c[2] == "en")
    audio = common.load_wav(path)
    t = np.arange(len(audio)) / common.SR
    ax.plot(t, audio, lw=0.5)
    ax.set_ylabel(label)
axes[-1].set_xlabel("seconds")
plt.tight_layout()
plt.show()

## Step 2. Turn audio into numbers (embeddings)

Raw audio is 48,000 numbers for 3 seconds. That is too many, and they mean little on their own.

**whisper-tiny** is a speech model from OpenAI. It was trained on 680,000 hours of speech
in many languages. We use only its **encoder** and we keep it **frozen**: we never change its weights.

- The encoder gives one 384-number vector for every 20 ms of audio.
- We **average** these vectors over the whole clip (what was said, in general).
- We also average over the **last 1 second** (how the clip ends - the key for turn-taking).
- Together: one vector of **768 numbers** per clip. This is the *embedding*.

In [ ]:
embed = common.Embedder()   # loads whisper-tiny from the models folder

audio = common.load_wav(clips[0][0])
vec = embed([audio])
print("Audio samples:", len(audio))
print("Embedding shape:", vec.shape)
print("First 8 numbers:", np.round(vec[0, :8], 3))

Now do this for every clip. The starter clips already have **precomputed** features,
so this is fast. Only your own new recordings (in the `data` folder) need the model.
On a slow laptop, new clips take about half a second each.

In [ ]:
data = common.build_features()
X, y, langs = data["X"], data["y"], data["langs"]
print("X shape:", X.shape, "  (clips x numbers)")
print("Clips per label:", dict(zip(common.LABELS, np.bincount(y))))

## Step 3. Logistic regression

Logistic regression draws straight lines (planes) between the classes. It is fast and
a good first try. We keep **20%** of the clips aside as a **test set**.
The model never sees them while training, so the test score is honest.

`StandardScaler` makes every one of the 768 numbers have mean 0 and spread 1.
This helps the classifier.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_tr, X_te, y_tr, y_te, l_tr, l_te = train_test_split(
    X, y, langs, test_size=0.2, stratify=y, random_state=42)

logreg = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=2000))
logreg.fit(X_tr, y_tr)
pred = logreg.predict(X_te)
print(f"Test accuracy: {accuracy_score(y_te, pred) * 100:.1f}%")

ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=common.LABELS, cmap="Blues")
plt.title("Logistic regression")
plt.show()

**How to read the confusion matrix:** rows are the truth, columns are the prediction.
Numbers on the diagonal are correct. Which two labels get mixed up most? Why do you think so?

Accuracy per language:

In [ ]:
for g in sorted(set(l_te)):
    m = l_te == g
    print(f"{common.LANGS.get(g, g):<8} {accuracy_score(y_te[m], pred[m]) * 100:5.1f}%  ({m.sum()} test clips)")

## Step 4. A tiny neural network (MLP)

Now a small neural network in PyTorch: 768 inputs -> 128 hidden units -> 3 outputs.
It can learn curved boundaries, not only straight ones.

- **Dropout** randomly switches off 30% of hidden units while training. This stops it from memorising.
- **Epoch** = one pass over all training clips.
- **Loss** should go down as it learns.

In [ ]:
import torch

torch.manual_seed(42)
mean, scale = X_tr.mean(0), X_tr.std(0) + 1e-6
xt, yt = torch.from_numpy((X_tr - mean) / scale), torch.from_numpy(y_tr)
xv = torch.from_numpy((X_te - mean) / scale)

net = common.make_mlp(X.shape[1], hidden=128)
print(net)
opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-3)
loss_fn = torch.nn.CrossEntropyLoss()

losses, accs = [], []
for epoch in range(300):
    net.train()
    perm = torch.randperm(len(xt))
    for i in range(0, len(xt), 64):
        idx = perm[i:i + 64]
        opt.zero_grad()
        loss = loss_fn(net(xt[idx]), yt[idx])
        loss.backward()
        opt.step()
    net.eval()
    with torch.no_grad():
        accs.append((net(xv).argmax(1).numpy() == y_te).mean())
    losses.append(loss.item())

print(f"MLP test accuracy: {accs[-1] * 100:.1f}%")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3))
a1.plot(losses); a1.set_title("training loss"); a1.set_xlabel("epoch")
a2.plot(accs); a2.set_title("test accuracy"); a2.set_xlabel("epoch")
plt.show()

In [ ]:
with torch.no_grad():
    mlp_pred = net(xv).argmax(1).numpy()
ConfusionMatrixDisplay.from_predictions(y_te, mlp_pred, display_labels=common.LABELS, cmap="Blues")
plt.title("MLP")
plt.show()

## Step 5. Try it with your voice

Run the cell, then speak for 3 seconds. Try:
- a full question: *"What time is the lecture?"*
- half a sentence, then pause: *"I was thinking that..."*
- a stop word: *"ruko"*, *"thamba"*, *"nillisu"*, *"stop"*

If the cell says "silence", allow mic access in Windows privacy settings.

In [ ]:
def predict(audio):
    x = embed([audio])
    with torch.no_grad():
        probs = torch.softmax(net(torch.from_numpy((x - mean) / scale)), -1)[0].numpy()
    return probs

print("Recording 3 seconds ... speak now!")
audio = common.record(3.0)
display(Audio(audio, rate=common.SR))
if not common.has_speech(audio):
    print("I heard only silence. Check your mic.")
else:
    probs = predict(audio)
    for label, p in zip(common.LABELS, probs):
        print(f"{label:<11} {'#' * int(p * 30):<30} {p * 100:5.1f}%")

## Step 6. Make it better (exercises)

1. The starter data is synthetic (TTS). Real voices sound different.
   Open a Command Prompt and run `python record.py`. Record **10 clips of each label**
   in your own language.
2. Re-run this notebook from **Step 2** (or run `python extract_features.py` then `python train.py`).
   Did the live test get better?
3. Change `C=0.1` in logistic regression to `1.0` or `0.01`. What happens to test accuracy?
4. Change `hidden=128` to `32` or `512`. Is bigger always better?
5. Which language is hardest? Why might that be?
6. Swap data with a friend who speaks a different language. Does your model work on their voice?

**Big idea:** we never trained the big speech model. We reused a frozen model and trained
only a tiny "head" on top. This is called **transfer learning**. It is how most real
speech and vision systems are built on small budgets.